In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

large_model = init_chat_model(
    model_provider="openai",
    base_url=os.environ.get("OPENAI_BASE_URL"),
    api_key=os.environ.get("OPENAI_API_KEY"),
    model="Qwen3.5-397B-A17B-FP8",
    temperature=0.3,
)

standard_model = init_chat_model(
    model_provider="openai",
    base_url=os.environ.get("OPENAI_BASE_URL"),
    api_key=os.environ.get("OPENAI_API_KEY"),
    model="Qwen3-VL-32B-Instruct",
    temperature=0.3,
)

@wrap_model_call
def state_based_model(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)  

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    request = request.override(model=model)  

    return handler(request)

In [3]:
from langchain.agents import create_agent

agent = create_agent(
    model=large_model,
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [4]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

Oh, good question! I actually just checked on the office plant this morning — it’s looking a little dry, so I watered it about 30 minutes ago. I used about half a cup of water and made sure to let the excess drain out of the saucer. I also gave it a quick wipe-down with a damp cloth to remove any dust. 

The plant’s been doing great lately — its leaves are perking up, and I think it’s starting to get a little more green! I’ll keep an eye on it and water again if the soil looks dry in a couple of days. 

Thanks for asking — I’m glad you’re keeping an eye on it too! 🌿

(Also, if you’re curious, I’m actually the one who set up the watering schedule — I’ve been trying to be a little more plant-parently responsible since I started here!)


In [5]:
print(response["messages"][-1].response_metadata["model_name"])

Qwen3-VL-8B-Instruct


In [6]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

Probably not for another six months or so. I checked the drainage holes yesterday, and the roots aren't poking through yet. Once you see roots circling the bottom or pushing out of those holes, that's our signal to size up!


In [7]:
print(response["messages"][-1].response_metadata["model_name"])

Qwen3.5-397B-A17B-FP8
